# Wafer Fabrication Log

Use this notebook to record a new wafer batch:
1. Photolithography (spin coat, laser writing, developing)
2. Cr/Au Evaporation
3. Lift-off
4. Cutting into pieces

Fill in the cells below, execute them, and your samples will be registered in the database.

In [1]:
import sys, os, json
from datetime import datetime
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, PROJECT_ROOT)

from src.database import (
    init_db, add_wafer, get_wafer, add_step,
    get_sample, list_wafers, list_samples
)
import pandas as pd

init_db()
print('Database ready.')

Database ready.


## 1. Wafer Info

Fill in the wafer details and run the cell to register it.

In [2]:
DATE = datetime.now().strftime('%y%m%d')  # e.g., '260626'
WAFER_NUM = 1  # Change per batch
SUBSTRATE = 'Kapton'  # Kapton, Si/SiO2, Glass
THICKNESS = '75um'
N_PIECES = 4
N_DEVICES_PER_PIECE = 3
NOTES = ''

wafer_name = f'{DATE}_W{WAFER_NUM}'
print(f'Wafer name: {wafer_name}')
print(f'Substrate: {SUBSTRATE} ({THICKNESS})')
print(f'Pieces: {N_PIECES} x {N_DEVICES_PER_PIECE} devices = {N_PIECES * N_DEVICES_PER_PIECE} total')

Wafer name: 260702_W1
Substrate: Kapton (75um)
Pieces: 4 x 3 devices = 12 total


In [3]:
# Run this to create the wafer in the DB
wafer_id = add_wafer(
    name=wafer_name,
    substrate=SUBSTRATE,
    size=THICKNESS,
    n_pieces=N_PIECES,
    n_devices_per_piece=N_DEVICES_PER_PIECE,
    notes=NOTES,
)

wf = get_wafer(wafer_id)
print(f'Created wafer {wf["name"]} (id={wafer_id})')
for s in wf['samples']:
    sample = get_sample(s['id'])
    devs = ', '.join(f'D{d["device_number"]}' for d in sample['devices'])
    print(f'  {sample["label"]} -> devices: {devs}')

Created wafer 260702_W1 (id=2)
  260702_W1_P1 -> devices: D1, D2, D3
  260702_W1_P2 -> devices: D1, D2, D3
  260702_W1_P3 -> devices: D1, D2, D3
  260702_W1_P4 -> devices: D1, D2, D3


## 2. Photolithography

Record parameters for the photolithography step.

In [4]:
PHOTO_PARAMS = {
    'resist': 'S1813',
    'spin_rpm': 4000,
    'spin_time_s': 25,
    'anneal_temp_C': 75,
    'anneal_time_min': 1,
    'exposure_focus_points': 9,
}
PHOTO_NOTES = ''

# Apply to all samples in this wafer
for s in wf['samples']:
    add_step(s['id'], 'photolithography', status='completed',
             params=PHOTO_PARAMS, notes=PHOTO_NOTES)
    
    add_step(s['id'], 'developing', status='completed',
             notes='Developer immersion 40-60s, rinse ultra-pure water, N2 dry')

print(f'Photolithography + developing registered for {len(wf["samples"])} samples.')

Photolithography + developing registered for 4 samples.


## 3. Evaporation

Cr/Au evaporation (done by cleanroom technicians).

In [5]:
EVAP_PARAMS = {'cr_nm': 5, 'au_nm': 40}
EVAP_NOTES = ''

for s in wf['samples']:
    add_step(s['id'], 'evaporation', status='completed',
             params=EVAP_PARAMS, notes=EVAP_NOTES)

print('Evaporation registered.')

Evaporation registered.


## 4. Lift-off

3x Acetone baths + 3x IPA baths (or 1x 20 min each).

In [6]:
LIFTOFF_PARAMS = {
    'acetone_baths': 3,
    'acetone_time_min': 15,
    'ipa_baths': 3,
    'ipa_time_min': 15,
}

for s in wf['samples']:
    add_step(s['id'], 'lift_off', status='completed',
             params=LIFTOFF_PARAMS, notes='Fresh acetone/IPA each bath.')

print('Lift-off registered.')

Lift-off registered.


## 5. Cutting

Cut wafer into 4 pieces. Mark as completed.

In [7]:
for s in wf['samples']:
    add_step(s['id'], 'cutting', status='completed',
             notes='Protect with wipe while cutting. Store in vacuum/glovebox.')

print('Cutting registered.')
print()
print('=== Wafer fabrication complete! ===')
print('Store samples in vacuum/glovebox until SAM + BAMS.')

Cutting registered.

=== Wafer fabrication complete! ===
Store samples in vacuum/glovebox until SAM + BAMS.


---
### Notes
- Modify parameters above before running each cell.
- Add photos by placing them in `data/photos/` and updating notes.
- Run the dashboard to see the sample status board: `streamlit run dashboard/app.py`